# Structure x function: reading the (k, phi) grid

`structure_function_grid.py` trains one model per cell of a two-axis grid and scores the
receptor embedding it produces. This notebook is the readout, and it computes nothing: it
reads `results/sf_grid/<dataset>/{grid.csv, nulls.csv, meta.json}` and draws them.

The two axes are the same kind of knob pointed at opposite sources -- each **destroys**
information rather than reweighting it, which is what lets the picture be read as a mix:

* **k** -- how many PCA components of ESM the receptor node features carry. `k = 0` is a
  one-hot identity (no structure at all); `k = all` is the full ESM space.
* **phi** -- the fraction of message-passing edges kept, stratified by sign. `phi = 0`
  removes message passing entirely; `phi = 1` is the full binding graph. Crucially the
  *supervision* is untouched at every phi -- `_mp_edges` drops molecules "from message
  passing only; every train row still gets decoded/supervised regardless" -- so less
  function is not confounded with less data.

Each cell's embedding is scored by three geometries of increasing strictness (RSA ->
CCA -> Procrustes) against three reference clouds. The first two are a **manipulation
check**, the third is the **result**:

| reference | question |
|---|---|
| `esm` | how structural did this come out |
| `func` | how functional did this come out |
| `target` | and is it any good on the held-out class |

**What would count as a win.** Not a high number anywhere -- a *shape*. If the aggregate of
two sources beats both pure sources, the target panel has an **interior maximum**: a cell
with neither k nor phi at an extreme scoring above both the (k=all, phi=0) and the
(k=0, phi=1) corners. That is what section 5 is for. A target surface that rises
monotonically toward one corner says the mix buys nothing, and is just as publishable an
answer.

Read section 1 first: the grid is filled trajectory by trajectory, so most runs will have a
sparse surface, and the 1-D arms of section 2 are then the honest reading.

In [ ]:
%matplotlib inline
import json, pathlib
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
try:
    from IPython.display import display
except ImportError:            # so the same code runs outside a notebook
    display = print

# ---------------------------------------------------------------- flags
DATASET = "hc"        # "hc" or "cc" -- insects only, by construction
CLASSES = None        # None = every held-out class present; or e.g. ["aromatic"]
VALUE   = "z"         # "z": dimensionless, vs each cell's OWN permutation null.
                      # "raw": the geometry's own units -- comparable within a column, NOT
                      # across geometries (CCA's null alone sits near 0.30 on 24 receptors)
# ----------------------------------------------------------------------

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RESULTS = ROOT / "results" / "sf_grid"

GEOMS = ["rsa", "cca", "procrustes"]
REFS = ["esm", "func", "target"]
KEY = ["k", "phi", "cls", "seed"]
REF_LABEL = {"esm": "vs raw ESM  (structure)", "func": "vs retained profile  (function)",
             "target": "vs held-out class  (result)"}
GEOM_LABEL = {"rsa": "RSA", "cca": "CCA", "procrustes": "Procrustes"}
# Okabe-Ito: distinguishable under every common colour-vision deficiency, in fixed order
CAT = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00"]
SEQ = LinearSegmentedColormap.from_list("seq", ["#f2f6fa", "#9ec5e0", "#3d8dbc", "#0b3f61"])
DIV = LinearSegmentedColormap.from_list("div", ["#0b3f61", "#79aecd", "#dcdcdc",
                                                "#e8a06a", "#8c3708"])
mpl.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
                     "font.size": 9, "axes.titlesize": 9.5})

# The arms `structure_function_grid.py --trajectory ...` actually walks. Named here rather
# than inferred, so a half-finished run still gets its arms drawn instead of a guess.
ARMS = {"k-arm  (phi = 1, full graph)": lambda k, p: p == 1.0,
        "phi-arm  (k = all, full ESM)": lambda k, p: k < 0,
        "diagonal  (trade one for the other)":
            lambda k, p: (k, p) in {(-1, 0.0), (32, 0.05), (8, 0.25), (2, 0.5), (0, 1.0)}}


def klab(k):
    return "all" if k < 0 else str(int(k))


def kord(vals):
    return sorted(set(vals), key=lambda k: (k < 0, k))     # 0, 2, ... 128, all


def load(ds):
    """grid.csv + nulls.csv -> one long frame, a row per (cell, seed, geometry, reference)."""
    d = RESULTS / ds
    gp, npth = d / "grid.csv", d / "nulls.csv"
    if not gp.exists():
        have = [p.name for p in RESULTS.glob("*")] if RESULTS.exists() else f"nothing under {RESULTS}"
        raise FileNotFoundError(f"{gp} not found. Available: {have}")
    grid = pd.read_csv(gp).drop_duplicates(subset=KEY, keep="last")
    extra = [c for c in ("n_pos", "n_neg", "k_eff", "dim_z") if c in grid.columns]
    parts = []
    for g in GEOMS:
        for r in REFS:
            col = f"{g}__{r}"
            if col not in grid.columns:
                continue
            s = grid[KEY + extra].copy()
            s["geom"], s["ref"], s["value"] = g, r, grid[col].to_numpy()
            parts.append(s)
    if not parts:
        raise ValueError(f"{gp} carries no <geometry>__<reference> columns")
    long = pd.concat(parts, ignore_index=True)

    if npth.exists():
        nul = pd.read_csv(npth).drop_duplicates(subset=["k", "phi", "cls", "ref"], keep="last")
        nparts = []
        for g in GEOMS:
            a, b = f"{g}_null", f"{g}_null_sd"
            if a not in nul.columns:
                continue
            s = nul[["k", "phi", "cls", "ref", a, b]].rename(columns={a: "null", b: "null_sd"})
            s["geom"] = g
            nparts.append(s)
        if nparts:
            # a null is computed once per cell (on the first seed); it attaches to every seed
            long = long.merge(pd.concat(nparts, ignore_index=True),
                              on=["k", "phi", "cls", "ref", "geom"], how="left")
    for c in ("null", "null_sd"):
        if c not in long.columns:
            long[c] = np.nan
    long["z"] = (long["value"] - long["null"]) / long["null_sd"].where(long["null_sd"] > 1e-12)
    mp = d / "meta.json"
    meta = json.loads(mp.read_text(encoding="utf-8")) if mp.exists() else {}
    return long, meta


LONG, META = load(DATASET)
if CLASSES:
    absent = sorted(set(CLASSES) - set(LONG["cls"]))
    if absent:
        print(f"!! not in this run, ignored: {absent}")
    LONG = LONG[LONG["cls"].isin(CLASSES)]
if LONG.empty:
    raise ValueError("no rows left after the class filter")

CELLS = sorted({(k, p) for k, p in zip(LONG["k"], LONG["phi"])}, key=lambda c: (c[0] < 0, c))
KS, PHIS = kord(LONG["k"]), sorted(set(LONG["phi"]))
NZ = int(LONG["z"].notna().sum())

print(f"{RESULTS / DATASET}")
print(f"  {LONG.groupby(KEY).ngroups} trainings"
      f" | classes {sorted(set(LONG['cls']))} | seeds {sorted(set(LONG['seed']))}")
print(f"  k    {[klab(k) for k in KS]}")
print(f"  phi  {[f'{p:g}' for p in PHIS]}")
print(f"  {len(CELLS)} of {len(KS) * len(PHIS)} (k, phi) combinations visited")
if META:
    v = META.get("variant", {})
    print(f"  meta: {META.get('epochs')} epochs, {v.get('criterion', '?')} q={v.get('q', '?')},"
          f" {META.get('n_perm')} permutations per null")
if NZ == 0:
    print("!! no usable nulls -- z is undefined everywhere, falling back to raw")
elif NZ < len(LONG):
    print(f"!! {len(LONG) - NZ} of {len(LONG)} readouts have no null (drawn blank)")
VALUE = VALUE if (VALUE == "raw" or NZ) else "raw"
VCOL = "z" if VALUE == "z" else "value"
VLAB = "z vs permutation null" if VALUE == "z" else "similarity (raw)"


def agg(df, by):
    """Mean over the (class, seed) trainings, with their spread. n < 2 -> no error bar."""
    out = df.groupby(by, dropna=False)[VCOL].agg(["mean", "std", "count"]).reset_index()
    out["err"] = np.where(out["count"] > 1, out["std"] / np.sqrt(out["count"]), np.nan)
    return out


def matrix(df, value=None):
    """(k, phi) matrix on the full observed axes; NaN where a cell was never run."""
    if df.empty:
        return pd.DataFrame(np.nan, index=KS, columns=PHIS)
    m = df.pivot_table(index="k", columns="phi", values=value or VCOL, aggfunc="mean")
    return m.reindex(index=KS, columns=PHIS)


def heat(ax, M, cmap, fmt="{:+.1f}", title="", **kw):
    A = np.ma.masked_invalid(np.asarray(M, dtype=float))
    im = ax.imshow(A, cmap=cmap, origin="lower", aspect="auto", **kw)
    ax.set_xticks(range(len(M.columns)), [f"{p:g}" for p in M.columns])
    ax.set_yticks(range(len(M.index)), [klab(k) for k in M.index])
    ax.set_title(title)
    ax.grid(False)
    lo, hi = im.get_clim()
    span = hi - lo if np.isfinite(hi - lo) and hi > lo else 1.0
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            if A.mask.ndim and A.mask[i, j]:
                ax.plot(j, i, marker="x", ms=4, mew=1, color="#c8c8c8")   # never run
                continue
            t = (float(A[i, j]) - lo) / span
            ax.text(j, i, fmt.format(float(A[i, j])), ha="center", va="center", fontsize=7,
                    color="white" if t > 0.62 else "#222222")
    return im

## 1. What has actually been computed

The grid is filled one trajectory at a time and every cell is written the moment it
finishes, so a partial surface is the normal state, not a failure. This is the map of it:
how many (class, seed) trainings stand behind each (k, phi) cell. An `x` is a cell that was
never run, and everything downstream draws those blank rather than interpolating them.

The right-hand panel is the audit of the phi knob itself -- the edge counts it left behind,
positive and negative separately, since the thinning is stratified by sign precisely so that
a small phi does not quietly turn into a one-signed graph.

In [ ]:
cnt = (LONG[(LONG.geom == GEOMS[0]) & (LONG.ref == REFS[0])]
       .pivot_table(index="k", columns="phi", values="value", aggfunc="count")
       .reindex(index=KS, columns=PHIS))
fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.2))
im = heat(axes[0], cnt, SEQ, fmt="{:.0f}", vmin=0, title="trainings per cell  (classes x seeds)")
axes[0].set_xlabel("phi   fraction of MP edges kept")
axes[0].set_ylabel("k   ESM components")
fig.colorbar(im, ax=axes[0], fraction=0.046)

ax = axes[1]
ec = (LONG.drop_duplicates(subset=["k", "phi", "cls", "seed"])
      .groupby("phi")[["n_pos", "n_neg"]].mean() if "n_pos" in LONG.columns else pd.DataFrame())
if not ec.empty:
    x = np.arange(len(ec))
    ax.bar(x - 0.2, ec["n_pos"], 0.38, color=CAT[0], label="positive edges")
    ax.bar(x + 0.2, ec["n_neg"], 0.38, color=CAT[1], label="negative edges")
    ax.set_xticks(x, [f"{p:g}" for p in ec.index])
    ax.set_xlabel("phi")
    ax.set_ylabel("message-passing edges")
    ax.set_title("what phi removed  (sign-stratified)")
    ax.legend(frameon=False, fontsize=8)
else:
    ax.set_axis_off()
fig.suptitle(f"{DATASET.upper()} -- coverage", y=1.02, fontsize=10.5)
plt.tight_layout(); plt.show()

gaps = [c for c in ((k, p) for k in KS for p in PHIS) if c not in set(CELLS)]
print(f"{len(CELLS)} cells present, {len(gaps)} unvisited on the observed axes")
if gaps:
    print("  unvisited:", ", ".join(f"(k={klab(k)}, phi={p:g})" for k, p in gaps[:24]),
          "..." if len(gaps) > 24 else "")

## 2. The arms: one knob at a time

The `edges` trajectory holds one source at full and sweeps the other, which is the cheapest
question worth asking -- *does this source help at all* -- and the one that survives a
half-finished grid. The `diagonal` then trades one for the other.

Zero is the permutation null and the band is +/- 2 of its sd, so a point inside the band is
a cloud with no alignment to that reference at all. Read the first two columns as the
manipulation check: the k-arm should slope up against ESM, the phi-arm up against the
profile. If they do not, the knobs are not doing what they were built to do and the third
column means nothing.

In [ ]:
for i_arm, (arm, sel) in enumerate(ARMS.items()):
    sub = LONG[[bool(sel(k, p)) for k, p in zip(LONG["k"], LONG["phi"])]]
    if sub.empty:
        print(f"-- {arm}: not run")
        continue
    xax = "k" if arm.startswith("k-arm") else "phi"   # the axis that moves along this arm
    if sub[xax].nunique() < 2 and sub["k"].nunique() > 1:
        xax = "k"
    order = kord(sub[xax]) if xax == "k" else sorted(set(sub[xax]))
    pos = {v: i for i, v in enumerate(order)}
    fig, axes = plt.subplots(1, 3, figsize=(11.2, 2.9))
    for j, r in enumerate(REFS):
        ax = axes[j]
        if VALUE == "z":
            ax.axhspan(-2, 2, color="#ededed", zorder=0, lw=0)
            ax.axhline(0, color="#9a9a9a", lw=0.9, zorder=1)
        for c, g in zip(CAT, GEOMS):
            a = agg(sub[(sub.ref == r) & (sub.geom == g)], [xax])
            if a.empty:
                continue
            a = a.sort_values(xax, key=lambda s: s.map(pos))
            ax.errorbar([pos[v] for v in a[xax]], a["mean"], yerr=a["err"], color=c,
                        marker="o", ms=4.5, lw=1.8, capsize=2.5, label=GEOM_LABEL[g])
        ax.set_xticks(range(len(order)),
                      [klab(v) if xax == "k" else f"{v:g}" for v in order])
        ax.set_xlabel("k   ESM components" if xax == "k" else "phi   MP edges kept")
        ax.set_title(REF_LABEL[r])
        if j == 0:
            ax.set_ylabel(VLAB)
    axes[0].legend(frameon=False, fontsize=8, ncol=3)
    fig.suptitle(f"{DATASET.upper()} -- {arm}", y=1.04, fontsize=10.5)
    plt.tight_layout(); plt.show()

## 3. The surface

The same numbers as a 3 x 3 panel of (k, phi) maps: one row per geometry, one column per
reference. The rightmost column is the only one that is a result; the left two say where
the knobs put the representation.

**Each panel carries its own colour scale**, so colour here means *shape within a panel* and
nothing else. That is deliberate: the ESM column runs to z ~ +15 while the target column
lives near single digits, and a shared scale would flatten the one column the grid was built
to read. The printed numbers are the comparable quantity -- and section 4 makes the
cross-reference comparison properly, as a difference.

In [ ]:
fig, axes = plt.subplots(len(GEOMS), len(REFS), figsize=(11.8, 3.0 * len(GEOMS)),
                         squeeze=False, layout="constrained")
for i, g in enumerate(GEOMS):
    row = LONG[LONG.geom == g]
    for j, r in enumerate(REFS):
        ax = axes[i][j]
        M = matrix(row[row.ref == r])
        v = np.asarray(M, float)
        v = v[np.isfinite(v)]
        lo, hi = (min(0.0, float(v.min())), float(v.max())) if v.size else (0.0, 1.0)
        im = heat(ax, M, SEQ, fmt="{:+.1f}" if VALUE == "z" else "{:.2f}",
                  title=REF_LABEL[r] if i == 0 else "",
                  vmin=lo, vmax=hi if hi > lo else lo + 1e-9)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
        if j == 0:
            ax.set_ylabel(f"{GEOM_LABEL[g]}\nk   ESM components")
        if i == len(GEOMS) - 1:
            ax.set_xlabel("phi   MP edges kept")
fig.suptitle(f"{DATASET.upper()} -- {VLAB}, mean over "
             f"{LONG['cls'].nunique()} class(es) x {LONG['seed'].nunique()} seed(s)",
             fontsize=10.5)
plt.show()

## 4. Where each cell landed

The manipulation check as one number: alignment with the profile minus alignment with ESM.
Blue is a representation that came out structural, orange one that came out functional, grey
one that is neither or both. A clean gradient across the diagonal is the evidence that the
two knobs really are opposed -- that the axes of section 3 mean what their names say.

In [ ]:
fig, axes = plt.subplots(1, len(GEOMS), figsize=(11.4, 3.3), squeeze=False)
for j, g in enumerate(GEOMS):
    row = LONG[LONG.geom == g]
    d = matrix(row[row.ref == "func"]) - matrix(row[row.ref == "esm"])
    a = np.asarray(d, float)
    v = float(np.nanmax(np.abs(a))) if np.isfinite(a).any() else 1.0
    im = heat(axes[0][j], d, DIV, vmin=-max(v, 1e-9), vmax=max(v, 1e-9), title=GEOM_LABEL[g])
    axes[0][j].set_xlabel("phi   MP edges kept")
    if j == 0:
        axes[0][j].set_ylabel("k   ESM components")
    fig.colorbar(im, ax=axes[0][j], fraction=0.046, label="structural  <->  functional")
fig.suptitle(f"{DATASET.upper()} -- position:  {VLAB} against the profile minus against ESM",
             y=1.03, fontsize=10.5)
plt.tight_layout(); plt.show()

## 5. The question: is there an interior maximum?

Every cell placed by *what it became* (x: functional minus structural, section 4) against
*how well it did* (y: alignment with the held-out class). The two pure corners are labelled
in bold.

**A hump between them is the result this grid was built to find.** It would say the best
representation of a receptor is neither its structure nor its measured function but a mix,
and say it as a shape rather than as which of two numbers is larger. A curve that climbs
monotonically to one edge says the opposite, just as cleanly.

In [ ]:
fig, axes = plt.subplots(1, len(GEOMS), figsize=(11.6, 3.5), squeeze=False, sharey=True)
for j, g in enumerate(GEOMS):
    ax = axes[0][j]
    row = LONG[LONG.geom == g]
    piv = {r: agg(row[row.ref == r], ["k", "phi"]).set_index(["k", "phi"]) for r in REFS}
    pts = []
    for (k, p) in CELLS:
        try:
            x = piv["func"].loc[(k, p), "mean"] - piv["esm"].loc[(k, p), "mean"]
            y, e = piv["target"].loc[(k, p), "mean"], piv["target"].loc[(k, p), "err"]
        except KeyError:
            continue
        if np.isfinite(x) and np.isfinite(y):
            pts.append((k, p, float(x), float(y), float(e) if np.isfinite(e) else 0.0))
    if not pts:
        ax.set_axis_off()
        continue
    for i, (arm, sel) in enumerate(ARMS.items()):
        q = sorted([t for t in pts if sel(t[0], t[1])], key=lambda t: t[2])
        if len(q) > 1:
            ax.plot([t[2] for t in q], [t[3] for t in q], color=CAT[i], lw=1.4, alpha=0.85,
                    zorder=1, label=arm.split("  ")[0])
    ax.errorbar([t[2] for t in pts], [t[3] for t in pts], yerr=[t[4] for t in pts],
                fmt="o", ms=5, color="#333333", ecolor="#bbbbbb", lw=0, elinewidth=1, zorder=3)
    # labels are "k|phi", offset alternately above and below along x, because the cells
    # bunch up hard at the functional end and a single offset makes that corner unreadable
    top = max(pts, key=lambda t: t[3])
    for i, (k, p, x, y, _) in enumerate(sorted(pts, key=lambda t: t[2])):
        key = ((k < 0 and p == 0.0) or (k == 0 and p == 1.0)
               or (k, p) == (top[0], top[1]))
        ax.annotate(f"{klab(k)}|{p:g}", (x, y), fontsize=6.5, textcoords="offset points",
                    xytext=(5, 5) if i % 2 == 0 else (5, -11), zorder=4,
                    color="#000000" if key else "#909090",
                    fontweight="bold" if key else "normal")
    if VALUE == "z":
        ax.axhline(0, color="#9a9a9a", lw=0.9)
    ax.axvline(0, color="#d0d0d0", lw=0.9)
    ax.set_title(GEOM_LABEL[g])
    ax.set_xlabel("structural   <-  position  ->   functional")
    if j == 0:
        ax.set_ylabel(f"alignment with the held-out class\n({VLAB})")
        ax.legend(frameon=False, fontsize=7.5, title="labels: k | phi", title_fontsize=7.5)
fig.suptitle(f"{DATASET.upper()} -- trajectory in representation space", y=1.03, fontsize=10.5)
plt.tight_layout(); plt.show()

## 6. The cells, as numbers

First the whole surface, then the ranking against the held-out class. `n` is the number of
(class, seed) trainings behind a row and `+/-` their standard error -- a cell standing on one
seed is a suggestion, not a finding.

In [ ]:
tab = agg(LONG, ["geom", "ref", "k", "phi"])
wide = tab.pivot_table(index=["k", "phi"], columns=["geom", "ref"], values="mean")
wide = wide.reindex(columns=pd.MultiIndex.from_product([GEOMS, REFS]))
wide.index = pd.MultiIndex.from_tuples([(klab(k), f"{p:g}") for k, p in wide.index],
                                       names=["k", "phi"])
print(f"the surface ({VLAB}):")
display(wide.round(2))

tgt = tab[tab.ref == "target"].sort_values("mean", ascending=False)
best = tgt.copy()
best["k"] = best["k"].map(klab)
best = best[["geom", "k", "phi", "mean", "err", "count"]].rename(
    columns={"mean": VALUE, "err": "+/-", "count": "n"})
print(f"\ntop cells against the held-out class, per geometry ({VLAB}):")
print(best.groupby("geom", sort=False).head(3).round(3).to_string(index=False))

# The verdict. The claim this grid exists to test is "a MIX beats both pure sources", so the
# comparison is against the two pure corners by name -- not against "an edge", since a cell
# like (k=32, phi=1) is partial ESM on a full graph and is already a mix.
PURE = {(-1.0, 0.0): "pure structure (k=all, phi=0)", (0.0, 1.0): "pure function (k=0, phi=1)"}
LABELS = {(0.0, 0.0): "the label-factorisation baseline (k=0, phi=0)"}
print("\nverdict:")
for g in GEOMS:
    q = tgt[tgt.geom == g]
    if q.empty:
        continue
    t = q.iloc[0]
    cell = (float(t["k"]), float(t["phi"]))
    have = {}
    for c, nm in PURE.items():
        hit = q[(q.k == c[0]) & (q.phi == c[1])]
        if not hit.empty:
            have[nm.split(" (")[0]] = float(hit["mean"].iloc[0])
    head = f"  {GEOM_LABEL[g]:<10} best k={klab(t['k'])}, phi={t['phi']:g}  ({t['mean']:+.2f})"
    if VALUE == "z" and t["mean"] < 2:
        print(f"{head}   -> nothing here clears its own null; no shape to read")
        continue
    if len(have) < 2:
        print(f"{head}   -> {2 - len(have)} of the 2 pure corners not run; cannot compare")
    elif cell in PURE:
        print(f"{head}   -> {PURE[cell]} wins: no mix beats it")
    elif cell in LABELS:
        print(f"{head}   -> {LABELS[cell]} wins -- see 'Reading this'")
    else:
        detail = ", ".join(f"{n} {v:+.2f}" for n, v in have.items())
        print(f"{head}   -> MIX WINS by {t['mean'] - max(have.values()):+.2f} "
              f"over the better pure source  [{detail}]")

## Reading this

**The manipulation check comes first.** If the k-arm does not climb against ESM and the
phi-arm does not climb against the retained profile, nothing in the target column is
interpretable -- the axes would not be the axes they are named for. Both were confirmed at 5
epochs on HC/aromatic before the grid was run: (k=all, phi=0) sits at z = +14.7 against ESM
and +0.6 against the target; (k=0, phi=1) at +0.6 against ESM and +2.7 against the profile.

**(k=0, phi=0) is not a null floor,** though it looks like the empty corner. The decoder
still supervises on every pair -- exactly the property that makes phi a clean knob -- so the
model there learns a plain factorisation of the training labels and scores well above zero
(z = +4.9 functional in the smoke). Read it as a third baseline, "labels without message
passing". The actual floor is the permutation null, which is why every cell carries its own
and why `VALUE = "z"` is the default.

**z is not an effect size across datasets.** The null's sd shrinks as the panel grows, so a z
of 6 on HC (24 receptors) and a z of 6 on CC (50) are not the same claim. Compare z within a
panel; switch to `VALUE = "raw"` to ask how big the thing actually is.

**One seed is not a result.** The `n` column in section 6 is the guard. Cold-molecule
holdouts on these panels swing by +/- 0.1 between seeds, and an interior maximum standing on
a single training is exactly the shape seed noise likes to produce.